# Giai đoạn 2: Tiền xử lý & Làm sạch dữ liệu PR (Data Preprocessing & Cleaning)
Dự án: **Phân tích Đối chiếu & Xây dựng Mô hình Cảnh báo Sớm cho Pull Request**
Thành viên thực hiện: **Trần Đức Thịnh (Data Analyst)**

Sổ tay này thực hiện các công việc:
- Kết nối tới cơ sở dữ liệu SQLite `github_prs.db`.
- Đọc dữ liệu từ View `view_pr_clean`.
- Phân tích và xử lý giá trị thiếu (Missing values).
- Tính toán lại thời gian sống thực tế của PR (`duration_minutes`) dựa trên hiệu số giữa `closed_at` và `created_at`.
- Chuẩn hóa định dạng thời gian (Datetime) trên Pandas.
- Loại bỏ các PR rác hoặc ngoại lệ (Outliers) dựa trên phân tích phân phối dữ liệu.
- Xuất dữ liệu sạch ra file `data/processed/clean_prs.csv`.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đường dẫn tương đối (tự động phát hiện thư mục làm việc)
current_dir = os.getcwd()
if os.path.basename(current_dir) == "notebooks":
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir

DB_PATH = os.path.join(BASE_DIR, "database", "github_prs.db")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "clean_prs.csv")

print(f"Base Directory: {BASE_DIR}")
print(f"Database Path: {DB_PATH}")
print(f"Output Path: {OUTPUT_PATH}")

### 1. Đọc dữ liệu từ Cơ sở dữ liệu SQLite
Chúng ta sẽ kết nối tới file DB SQLite và truy vấn toàn bộ dữ liệu từ View `view_pr_clean`.

In [ ]:
# Kết nối tới SQLite
conn = sqlite3.connect(DB_PATH)

# Đọc dữ liệu từ View view_pr_clean
df = pd.read_sql_query("SELECT * FROM view_pr_clean", conn)
conn.close()

# Hiển thị thông tin tổng quan
print(f"Kích thước dữ liệu gốc: {df.shape}")
df.head()

### 2. Phân tích và xử lý các dòng bị thiếu (Missing Values)
Chúng ta sẽ kiểm tra xem có trường dữ liệu nào bị khuyết thiếu (NULL) trong View hay không và đưa ra phương án xử lý thích hợp.

In [ ]:
# Kiểm tra số lượng giá trị thiếu trong mỗi cột
print("Thống kê số lượng giá trị thiếu (NaN):")
missing_summary = df.isna().sum()
print(missing_summary[missing_summary > 0])

**Nhận xét:**
Trong View `view_pr_clean`, chỉ có cột `merged_datetime` là chứa giá trị thiếu (151 dòng). Điều này hoàn toàn hợp lý vì đây là những PR **bị đóng mà không được merge (Rejected)**, do đó chúng không có mốc thời gian merge. Các cột khác như `title` hay `body_len` đã được xử lý bằng hàm `COALESCE` nên không còn giá trị NULL.

### 3. Chuẩn hóa định dạng thời gian (Datetime) trên Pandas & Tính toán lại thời gian sống của PR
Hiện tại, cột `duration_minutes` trong DB đang bị gán cứng bằng `0.0` đối với các PR bị từ chối (do không có `merged_at`). Tuy nhiên, thời gian sống thực tế của PR bị từ chối vẫn cần được tính từ lúc mở (`created_datetime`) đến lúc đóng (`closed_datetime`). Điều này sẽ giúp mô hình phân loại học được hành vi thực tế (ví dụ: PR bị "ngâm" quá lâu thì có dễ bị reject hay không).

In [ ]:
# Chuyển đổi các cột thời gian sang kiểu Datetime của Pandas
df['created_datetime'] = pd.to_datetime(df['created_datetime'])
df['closed_datetime'] = pd.to_datetime(df['closed_datetime'])
df['merged_datetime'] = pd.to_datetime(df['merged_datetime'])

# Tính toán thời gian sống thực tế (duration_minutes) theo phút
df['duration_minutes'] = (df['closed_datetime'] - df['created_datetime']).dt.total_seconds() / 60.0
df['duration_hours'] = df['duration_minutes'] / 60.0

# Kiểm tra xem có PR nào có duration_minutes < 0 (lỗi logic thời gian) không
negative_durations = df[df['duration_minutes'] < 0]
print(f"Số lượng PR có thời gian sống âm: {len(negative_durations)}")

print("\nThống kê thời gian sống thực tế (duration_minutes):")
print(df['duration_minutes'].describe())

### 4. Loại bỏ các PR rác hoặc ngoại lệ (Outliers)
Trong dữ liệu phần mềm, các chỉ số như `additions`, `deletions`, `changed_files`, và `commits` thường có phân phối lệch phải rất mạnh (Power-law distribution) với các giá trị cực đại siêu lớn (ví dụ: một PR sửa đổi hàng triệu dòng code hoặc hàng nghìn file). Những PR này thường đại diện cho các tác vụ hệ thống (nhập thư viện, cấu hình, build code tự động) chứ không phản ánh đúng quy trình phát triển thông thường của lập trình viên hoặc sinh viên, do đó cần được lọc bỏ để tránh làm sai lệch mô hình Hồi quy Logistic.

In [ ]:
# Vẽ biểu đồ phân phối trước khi lọc outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df, x='is_fpt', y='additions', ax=axes[0, 0]).set_title('Additions trước khi lọc')
axes[0, 0].set_yscale('log')

sns.boxplot(data=df, x='is_fpt', y='deletions', ax=axes[0, 1]).set_title('Deletions trước khi lọc')
axes[0, 1].set_yscale('log')

sns.boxplot(data=df, x='is_fpt', y='changed_files', ax=axes[1, 0]).set_title('Changed Files trước khi lọc')
axes[1, 0].set_yscale('log')

sns.boxplot(data=df, x='is_fpt', y='commits', ax=axes[1, 1]).set_title('Commits trước khi lọc')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

Chúng ta sẽ áp dụng các ngưỡng lọc ngoại lệ dựa trên kiến thức miền (domain knowledge) về các PR thông thường để giữ lại dữ liệu chất lượng nhất:
- `additions <= 5000` (PR sửa đổi không quá 5000 dòng code thêm mới).
- `deletions <= 5000` (PR sửa đổi không quá 5000 dòng code bị xóa).
- `changed_files <= 50` (PR không ảnh hưởng quá 50 file để đảm bảo tính cục bộ).
- `commits <= 50` (PR không chứa quá 50 commits).

In [ ]:
# Định nghĩa các ngưỡng lọc ngoại lệ
MAX_ADDITIONS = 5000
MAX_DELETIONS = 5000
MAX_CHANGED_FILES = 50
MAX_COMMITS = 50

# Lọc dữ liệu
df_clean = df[
    (df['additions'] <= MAX_ADDITIONS) &
    (df['deletions'] <= MAX_DELETIONS) &
    (df['changed_files'] <= MAX_CHANGED_FILES) &
    (df['commits'] <= MAX_COMMITS)
].copy()

print(f"Kích thước dữ liệu sau khi lọc outliers: {df_clean.shape}")
print(f"Số lượng dòng bị loại bỏ: {len(df) - len(df_clean)} ({((len(df) - len(df_clean))/len(df))*100:.2f}%)")

# Thống kê phân bổ lớp sau khi lọc
print("\nPhân bổ dữ liệu FPT vs Global sau khi làm sạch:")
print(df_clean['is_fpt'].value_counts().rename({1: 'FPT University', 0: 'Global PR'}))

### 5. Xuất dữ liệu sạch ra file CSV
Chúng ta sẽ tạo thư mục `data/processed/` nếu chưa có và lưu file dữ liệu sạch `clean_prs.csv` tại đó. File này sẽ được sử dụng trực tiếp để huấn luyện mô hình ở Giai đoạn 4.

In [ ]:
# Tạo thư mục đầu ra nếu chưa tồn tại
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Lưu ra file CSV (không lưu chỉ mục dòng)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"🎉 Đã lưu thành công dữ liệu sạch ({df_clean.shape[0]} dòng) tại: {OUTPUT_PATH}")